# TP J3 – UC-06 · Étape 3 : exploration des sources avec PySpark

**Retail Intelligence Platform** · Mastère Data & IA · 18/11/2026

**Objectif.** Lire les 4 sources retenues **directement dans HDFS** (zone Raw déposée au J2), puis observer le schéma,
un échantillon, le volume, les valeurs manquantes, les doublons sur la clé candidate et les valeurs atypiques.
Ce notebook **ne transforme rien** : il sert à décider des règles de nettoyage (étape 4).

| Source | Fichier HDFS | Rôle |
|---|---|---|
| `store_sales` | `/retail-lake/raw/tpcxbb/initial/store_sales/store_sales.dat` | Faits : lignes de ticket en magasin |
| `item` | `.../item/item.dat` | Dimension produit |
| `date_dim` | `.../date_dim/date_dim.dat` | Dimension date |
| `store` | `.../store/store.dat` | Dimension magasin |

Questions métier visées (J1) : **Q1** évolution du CA, **Q3** produits/catégories qui vendent le plus, **Q5** comparaison des magasins.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../transformations"))
from pyspark.sql import functions as F
import common as C

spark = C.spark_session("TP3-Exploration")
print("Spark", spark.version, "| master", spark.sparkContext.master, "| Python", sys.version.split()[0])
print("Source Raw :", C.RAW)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 15:34:06 WARN Utils: Your hostname, Meddy, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/18 15:34:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 15:34:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.3.0 | master local[*] | Python 3.12.3
Source Raw : hdfs://localhost:9000/retail-lake/raw/tpcxbb/initial


## 1. Première lecture « naïve » : ce que Spark propose avec `inferSchema`

Les fichiers `.dat` n'ont **pas d'en-tête** et utilisent le séparateur `|`. Avec `inferSchema=True`, Spark nomme les
colonnes `_c0`, `_c1`… et devine un type. On regarde ce qu'il propose pour `store_sales`, sans s'y fier.

In [2]:
naif = (spark.read.option("header", False).option("sep", "|").option("inferSchema", True)
        .csv(C.raw_path("store_sales")))
naif.printSchema()
naif.show(5)
print("Colonnes :", len(naif.columns))

root
 |-- _c0: integer (nullable = true)
 |-- _c1: integer (nullable = true)
 |-- _c2: integer (nullable = true)
 |-- _c3: integer (nullable = true)
 |-- _c4: integer (nullable = true)
 |-- _c5: integer (nullable = true)
 |-- _c6: integer (nullable = true)
 |-- _c7: integer (nullable = true)
 |-- _c8: integer (nullable = true)
 |-- _c9: integer (nullable = true)
 |-- _c10: integer (nullable = true)
 |-- _c11: double (nullable = true)
 |-- _c12: double (nullable = true)
 |-- _c13: double (nullable = true)
 |-- _c14: double (nullable = true)
 |-- _c15: double (nullable = true)
 |-- _c16: double (nullable = true)
 |-- _c17: double (nullable = true)
 |-- _c18: double (nullable = true)
 |-- _c19: double (nullable = true)
 |-- _c20: double (nullable = true)
 |-- _c21: double (nullable = true)
 |-- _c22: double (nullable = true)



+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|  _c0|  _c1|  _c2|  _c3|    _c4| _c5|  _c6|_c7|_c8|_c9|_c10| _c11|  _c12| _c13|   _c14|   _c15|   _c16|   _c17|  _c18|  _c19|   _c20|   _c21|   _c22|
+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|36890|71480|10001|44404| 493944|2898|44093| 10|294|  0|  31| 18.0| 30.78|23.08|  238.7| 715.48|  558.0| 954.18| 21.46|   0.0| 715.48| 736.94| 157.48|
|36890|71480|10000|44404| 493944|2898|44093| 10|279|  0|  54|99.21|148.82|83.34|3535.92|4500.36|5357.34|8036.28|225.02|   0.0|4500.36|4725.38|-856.98|
|36890|71480|15556|44404| 493944|2898|44093| 10|114|  0|  39|79.49| 94.59|83.24| 442.65|3246.36|3100.11|3689.01|184.07|616.81|2629.55|2813.62|-470.56|
|36890|45392| 4597|89832|1774847|6266|16285|  4|285|  1|  20|74.38|136.12|133.4|   54.4| 2668.

**Constat.** 23 colonnes, ce qui correspond au schéma TPCx-BB de `store_sales` (vérifié au J1). Spark propose des
`int` pour les clés et des `double` pour les montants : un `double` est approximatif pour de l'argent, on retiendra
des `decimal(7,2)`. Les noms `_c0…` sont incompréhensibles : on applique les noms officiels.

## 2. Lecture contrôlée : noms officiels, tout en texte

On lit chaque fichier avec un schéma **explicite en `string`** (`common.read_raw`). Ainsi aucune valeur n'est
convertie en silence : on mesure ensuite combien de valeurs **ne se convertissent pas** dans le type cible.

In [3]:
dfs = {t: C.read_raw(spark, t).cache() for t in C.SOURCES}
for t, df in dfs.items():
    print(f"{t:12s} lignes = {df.count():>9,d} | colonnes = {len(df.columns)}")

store_sales  lignes =   667,579 | colonnes = 23


item         lignes =    17,820 | colonnes = 22


date_dim     lignes =   109,573 | colonnes = 28


store        lignes =        12 | colonnes = 29


In [4]:
dfs["store_sales"].show(5, truncate=False)
dfs["item"].select("i_item_sk", "i_item_id", "i_rec_start_date", "i_rec_end_date", "i_category", "i_class",
                   "i_brand", "i_current_price").show(8, truncate=False)
dfs["store"].select("s_store_sk", "s_store_id", "s_store_name", "s_rec_start_date", "s_rec_end_date",
                    "s_closed_date_sk", "s_city", "s_state", "s_number_employees").show(12, truncate=False)
dfs["date_dim"].select("d_date_sk", "d_date", "d_year", "d_moy", "d_dom", "d_day_name", "d_weekend").show(5)

+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+-------------------+-------------+
|ss_sold_date_sk|ss_sold_time_sk|ss_item_sk|ss_customer_sk|ss_cdemo_sk|ss_hdemo_sk|ss_addr_sk|ss_store_sk|ss_promo_sk|ss_ticket_number|ss_quantity|ss_wholesale_cost|ss_list_price|ss_sales_price|ss_ext_discount_amt|ss_ext_sales_price|ss_ext_wholesale_cost|ss_ext_list_price|ss_ext_tax|ss_coupon_amt|ss_net_paid|ss_net_paid_inc_tax|ss_net_profit|
+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+

+---------+----------------+----------------+--------------+-----------------+---------------------+--------------------------------------------------+---------------+
|i_item_sk|i_item_id       |i_rec_start_date|i_rec_end_date|i_category       |i_class              |i_brand                                           |i_current_price|
+---------+----------------+----------------+--------------+-----------------+---------------------+--------------------------------------------------+---------------+
|0        |AAAAAAAAAAAAAAAA|2000-01-14      |NULL          |Sports & Outdoors|Fan Shop             |71TS7NSbvH1YbdiQMG6ttBHKAljiNoIRB                 |72.29          |
|1        |AAAAAAAAAAAAAAAB|2000-09-23      |NULL          |Books            |New                  |qNBimHGBu763udH                                   |95.67          |
|2        |AAAAAAAAAAAAAAAC|2000-09-01      |NULL          |Books            |Education & Reference|ma5                                               |62.74    

+----------+----------------+------------+----------------+--------------+----------------+------------+-------+------------------+
|s_store_sk|s_store_id      |s_store_name|s_rec_start_date|s_rec_end_date|s_closed_date_sk|s_city      |s_state|s_number_employees|
+----------+----------------+------------+----------------+--------------+----------------+------------+-------+------------------+
|0         |AAAAAAAAAAAAAAAA|Forest      |2000-08-02      |NULL          |71238           |Post Oak    |CO     |217               |
|1         |AAAAAAAAAAAAAAAB|Hillside    |2000-07-03      |NULL          |18994           |Jerusalem   |TN     |244               |
|2         |AAAAAAAAAAAAAAAC|Sherwood    |2000-09-01      |NULL          |8900            |Mount Vernon|CA     |266               |
|3         |AAAAAAAAAAAAAAAD|Pioneer     |2000-06-02      |NULL          |69902           |Austin      |TX     |221               |
|4         |AAAAAAAAAAAAAAAE|Elmwood     |2000-07-03      |NULL          |79

+---------+----------+------+-----+-----+----------+---------+
|d_date_sk|    d_date|d_year|d_moy|d_dom|d_day_name|d_weekend|
+---------+----------+------+-----+-----+----------+---------+
|        0|1900-01-01|  1900|    1|    1|    Monday|        N|
|        1|1900-01-02|  1900|    1|    2|   Tuesday|        N|
|        2|1900-01-03|  1900|    1|    3| Wednesday|        N|
|        3|1900-01-04|  1900|    1|    4|  Thursday|        N|
|        4|1900-01-05|  1900|    1|    5|    Friday|        N|
+---------+----------+------+-----+-----+----------+---------+
only showing top 5 rows


## 3. Profil de chaque source

Pour chaque colonne : valeurs **nulles** (champ vide dans le `.dat`) et valeurs **non convertibles** dans le type
cible (texte présent mais `cast` impossible). Pour la clé candidate : nombre de **doublons**.

In [5]:
def profil(t):
    df = dfs[t]
    n = df.count()
    exprs = []
    for c, typ in C.SOURCES[t]:
        exprs.append(F.sum(F.col(c).isNull().cast("int")).alias(f"null__{c}"))
        if typ != "string":
            # try_cast : en mode ANSI (Spark 4), cast() lèverait une erreur sur la première valeur invalide
            exprs.append(F.sum((F.col(c).isNotNull() & F.col(c).try_cast(typ).isNull()).cast("int")).alias(f"cast__{c}"))
    r = df.agg(*exprs).first().asDict()
    cols = []
    for c, typ in C.SOURCES[t]:
        cols.append({"colonne": c, "type_cible": typ, "nulls": r[f"null__{c}"],
                     "pct_null": round(100 * r[f"null__{c}"] / n, 3),
                     "non_convertibles": r.get(f"cast__{c}", 0)})
    k = C.KEYS[t]
    distinct = df.select(*k).distinct().count()
    key_null = df.filter(" OR ".join(f"{c} IS NULL" for c in k)).count()
    return {"table": t, "lignes": n, "colonnes": len(df.columns), "cle": k, "cle_distinctes": distinct,
            "doublons_cle": n - distinct, "cle_nulle": key_null, "detail": cols}

profils = {t: profil(t) for t in C.SOURCES}
for t, p in profils.items():
    print(f"\n=== {t} : {p['lignes']:,d} lignes, clé {p['cle']} -> {p['doublons_cle']:,d} doublon(s), "
          f"{p['cle_nulle']} clé(s) nulle(s)")
    for d in p["detail"]:
        if d["nulls"] or d["non_convertibles"]:
            print(f"   {d['colonne']:24s} {d['type_cible']:13s} nulls={d['nulls']:>7,d} ({d['pct_null']:.3f} %)"
                  f"  non_convertibles={d['non_convertibles']}")


=== store_sales : 667,579 lignes, clé ['ss_ticket_number', 'ss_item_sk'] -> 4,136 doublon(s), 0 clé(s) nulle(s)
   ss_cdemo_sk              int           nulls=    238 (0.036 %)  non_convertibles=0
   ss_hdemo_sk              int           nulls=    295 (0.044 %)  non_convertibles=0
   ss_addr_sk               int           nulls=    448 (0.067 %)  non_convertibles=0
   ss_promo_sk              int           nulls=    158 (0.024 %)  non_convertibles=0

=== item : 17,820 lignes, clé ['i_item_sk'] -> 0 doublon(s), 0 clé(s) nulle(s)
   i_rec_end_date           date          nulls= 17,820 (100.000 %)  non_convertibles=0
   i_item_desc              string        nulls=      4 (0.022 %)  non_convertibles=0
   i_brand_id               int           nulls=      8 (0.045 %)  non_convertibles=0
   i_brand                  string        nulls=      6 (0.034 %)  non_convertibles=0

=== date_dim : 109,573 lignes, clé ['d_date_sk'] -> 0 doublon(s), 0 clé(s) nulle(s)

=== store : 12 lignes, clé ['s_

## 4. Contrôles métier sur `store_sales` (le fait)

Grain attendu : **une ligne = un article d'un ticket**. On vérifie les doublons exacts, les quantités, les montants
négatifs et la cohérence des calculs `ext_sales_price = sales_price × quantity` et
`net_paid = ext_sales_price − coupon_amt`.

In [6]:
ss = dfs["store_sales"]
typ = dict(C.SOURCES["store_sales"])
sst = ss.select(*[F.col(c).try_cast(typ[c]).alias(c) for c in ss.columns])

n = sst.count()
exacts = n - sst.distinct().count()
cle = ["ss_ticket_number", "ss_item_sk"]
dups = sst.groupBy(*cle).count().filter("count > 1")
nb_groupes = dups.count()
nb_lignes_dup = dups.agg(F.sum("count")).first()[0] or 0
print(f"Lignes : {n:,d} | doublons exacts (toutes colonnes) : {exacts:,d}")
print(f"Clés (ticket, article) en double : {nb_groupes:,d} groupes, {nb_lignes_dup:,d} lignes concernées "
      f"({100*nb_lignes_dup/n:.2f} %)")

Lignes : 667,579 | doublons exacts (toutes colonnes) : 0
Clés (ticket, article) en double : 4,038 groupes, 8,174 lignes concernées (1.22 %)


In [7]:
# exemple de doublon de clé : les deux lignes sont-elles identiques ?
ex = dups.orderBy("ss_ticket_number").first()
sst.filter((F.col("ss_ticket_number") == ex.ss_ticket_number) & (F.col("ss_item_sk") == ex.ss_item_sk)) \
   .select("ss_ticket_number", "ss_item_sk", "ss_sold_date_sk", "ss_sold_time_sk", "ss_store_sk",
           "ss_quantity", "ss_sales_price", "ss_net_paid").show(truncate=False)

+----------------+----------+---------------+---------------+-----------+-----------+--------------+-----------+
|ss_ticket_number|ss_item_sk|ss_sold_date_sk|ss_sold_time_sk|ss_store_sk|ss_quantity|ss_sales_price|ss_net_paid|
+----------------+----------+---------------+---------------+-----------+-----------+--------------+-----------+
|44              |10003     |36890          |27946          |5          |31         |53.16         |1647.96    |
|44              |10003     |36890          |27946          |5          |46         |1.60          |5.89       |
+----------------+----------+---------------+---------------+-----------+-----------+--------------+-----------+



In [8]:
regles = {
    "quantite_nulle":            F.col("ss_quantity").isNull(),
    "quantite_<=0":              F.col("ss_quantity") <= 0,
    "date_vente_nulle":          F.col("ss_sold_date_sk").isNull(),
    "magasin_nul":               F.col("ss_store_sk").isNull(),
    "article_nul":               F.col("ss_item_sk").isNull(),
    "net_paid_nul":              F.col("ss_net_paid").isNull(),
    "net_paid_<0":               F.col("ss_net_paid") < 0,
    "sales_price_<0":            F.col("ss_sales_price") < 0,
    "net_profit_<0 (marge)":     F.col("ss_net_profit") < 0,
    "sales_price>list_price":    F.col("ss_sales_price") > F.col("ss_list_price"),
    "ext_sales!=price*qty":      F.abs(F.col("ss_ext_sales_price") - F.col("ss_sales_price") * F.col("ss_quantity")) > 0.01,
    "net_paid!=ext-coupon":      F.abs(F.col("ss_net_paid") - (F.col("ss_ext_sales_price") - F.col("ss_coupon_amt"))) > 0.01,
}
res = sst.agg(*[F.sum(c.cast("int")).alias(k) for k, c in regles.items()]).first().asDict()
controles_ss = {k: {"lignes": v or 0, "pct": round(100 * (v or 0) / n, 3)} for k, v in res.items()}
for k, v in controles_ss.items():
    print(f"{k:26s} {v['lignes']:>8,d}  ({v['pct']:.3f} %)")

quantite_nulle                    0  (0.000 %)
quantite_<=0                      0  (0.000 %)
date_vente_nulle                  0  (0.000 %)
magasin_nul                       0  (0.000 %)
article_nul                       0  (0.000 %)
net_paid_nul                      0  (0.000 %)
net_paid_<0                       0  (0.000 %)
sales_price_<0                    0  (0.000 %)
net_profit_<0 (marge)       495,971  (74.294 %)
sales_price>list_price            0  (0.000 %)
ext_sales!=price*qty              0  (0.000 %)
net_paid!=ext-coupon              0  (0.000 %)


In [9]:
sst.select(F.min("ss_quantity"), F.max("ss_quantity"), F.min("ss_sales_price"), F.max("ss_sales_price"),
           F.min("ss_net_paid"), F.max("ss_net_paid"), F.round(F.avg("ss_net_paid"), 2).alias("avg_net_paid"),
           F.min("ss_net_profit"), F.max("ss_net_profit")).show(truncate=False)
# valeurs atypiques : règle de Tukey (Q3 + 3*IQR) sur net_paid
q1, q3 = sst.approxQuantile("ss_net_paid", [0.25, 0.75], 0.001)
seuil = q3 + 3 * (q3 - q1)
atyp = sst.filter(F.col("ss_net_paid") > seuil).count()
print(f"net_paid : Q1={q1}, Q3={q3}, seuil Tukey Q3+3*IQR={seuil:.2f} -> {atyp:,d} lignes au-dessus ({100*atyp/n:.2f} %)")

+----------------+----------------+-------------------+-------------------+----------------+----------------+------------+------------------+------------------+
|min(ss_quantity)|max(ss_quantity)|min(ss_sales_price)|max(ss_sales_price)|min(ss_net_paid)|max(ss_net_paid)|avg_net_paid|min(ss_net_profit)|max(ss_net_profit)|
+----------------+----------------+-------------------+-------------------+----------------+----------------+------------+------------------+------------------+
|1               |100             |0.00               |199.30             |0.00            |18745.00        |1723.44     |-9866.60          |9134.00           |
+----------------+----------------+-------------------+-------------------+----------------+----------------+------------+------------------+------------------+



net_paid : Q1=245.27, Q3=2359.41, seuil Tukey Q3+3*IQR=8701.83 -> 11,902 lignes au-dessus (1.78 %)


## 5. Intégrité référentielle (jointures possibles ?)

Avant toute jointure, on vérifie que les clés étrangères du fait existent dans les dimensions. Une colonne au nom
proche ne suffit pas : on compte les lignes **non rapprochées**.

In [10]:
it = dfs["item"].select(F.col("i_item_sk").cast("int").alias("k")).distinct()
dd = dfs["date_dim"].select(F.col("d_date_sk").cast("int").alias("k")).distinct()
st = dfs["store"].select(F.col("s_store_sk").cast("int").alias("k")).distinct()

def orphelins(col, dim):
    base = sst.filter(F.col(col).isNotNull())
    return base.join(dim, base[col] == dim.k, "left_anti").count()

integrite = {
    "ss_item_sk -> item.i_item_sk": orphelins("ss_item_sk", it),
    "ss_sold_date_sk -> date_dim.d_date_sk": orphelins("ss_sold_date_sk", dd),
    "ss_store_sk -> store.s_store_sk": orphelins("ss_store_sk", st),
}
for k, v in integrite.items():
    print(f"{k:40s} orphelins = {v:,d}")

ss_item_sk -> item.i_item_sk             orphelins = 0
ss_sold_date_sk -> date_dim.d_date_sk    orphelins = 0
ss_store_sk -> store.s_store_sk          orphelins = 0


In [11]:
# période couverte par les ventes, et lignes par magasin
d = dfs["date_dim"].select(F.col("d_date_sk").cast("int").alias("d_date_sk"), F.col("d_date").cast("date").alias("d_date"),
                           F.col("d_year").cast("int").alias("d_year"))
v = sst.join(d, sst.ss_sold_date_sk == d.d_date_sk, "inner")
v.select(F.min("d_date").alias("premiere_vente"), F.max("d_date").alias("derniere_vente")).show()
v.groupBy("d_year").agg(F.count("*").alias("lignes"), F.round(F.sum("ss_net_paid"), 2).alias("ca_net_paid")) \
 .orderBy("d_year").show()
sst.groupBy("ss_store_sk").count().orderBy("ss_store_sk").show(20)

+--------------+--------------+
|premiere_vente|derniere_vente|
+--------------+--------------+
|    2001-01-01|    2005-12-13|
+--------------+--------------+



+------+------+------------+
|d_year|lignes| ca_net_paid|
+------+------+------------+
|  2001|134873|232498659.83|
|  2002|135084|230947265.82|
|  2003|134773|233575185.63|
|  2004|136014|234344934.29|
|  2005|126835|219163540.80|
+------+------+------------+



+-----------+-----+
|ss_store_sk|count|
+-----------+-----+
|          0|56018|
|          1|55676|
|          2|54633|
|          3|55397|
|          4|56211|
|          5|55122|
|          6|54574|
|          7|57589|
|          8|55737|
|          9|56040|
|         10|55109|
|         11|55473|
+-----------+-----+



## 6. Dimensions : unicité et historisation

`item` et `store` sont des dimensions à **historique** (colonnes `rec_start_date` / `rec_end_date`) : une même clé
métier (`i_item_id`, `s_store_id`) peut avoir plusieurs versions, chacune avec sa propre clé technique `*_sk`.

In [12]:
itm = dfs["item"]
print("item : i_item_sk distincts =", itm.select("i_item_sk").distinct().count(),
      "| i_item_id distincts =", itm.select("i_item_id").distinct().count(),
      "| versions courantes (rec_end_date vide) =", itm.filter("i_rec_end_date IS NULL").count())
itm.groupBy("i_category").count().orderBy("i_category").show(truncate=False)
sto = dfs["store"]
print("store : s_store_sk distincts =", sto.select("s_store_sk").distinct().count(),
      "| s_store_id distincts =", sto.select("s_store_id").distinct().count())
sto.select("s_store_sk", "s_store_id", "s_rec_start_date", "s_rec_end_date", "s_closed_date_sk").orderBy("s_store_id").show()

item : i_item_sk distincts = 17820 | i_item_id distincts = 17820 | versions courantes (rec_end_date vide) = 17820


+------------------------+-----+
|i_category              |count|
+------------------------+-----+
|Books                   |5335 |
|Clothing & Accessories  |610  |
|Electronics             |692  |
|Home & Kitchen          |476  |
|Movies & TV             |1286 |
|Music                   |8143 |
|Sports & Outdoors       |465  |
|Tools & Home Improvement|408  |
|Toys & Games            |405  |
+------------------------+-----+



store : s_store_sk distincts = 12 | s_store_id distincts = 12
+----------+----------------+----------------+--------------+----------------+
|s_store_sk|      s_store_id|s_rec_start_date|s_rec_end_date|s_closed_date_sk|
+----------+----------------+----------------+--------------+----------------+
|         0|AAAAAAAAAAAAAAAA|      2000-08-02|          NULL|           71238|
|         1|AAAAAAAAAAAAAAAB|      2000-07-03|          NULL|           18994|
|         2|AAAAAAAAAAAAAAAC|      2000-09-01|          NULL|            8900|
|         3|AAAAAAAAAAAAAAAD|      2000-06-02|          NULL|           69902|
|         4|AAAAAAAAAAAAAAAE|      2000-07-03|          NULL|           79617|
|         5|AAAAAAAAAAAAAAAF|      2000-03-03|          NULL|           88139|
|         6|AAAAAAAAAAAAAAAG|      2000-01-03|          NULL|           15761|
|         7|AAAAAAAAAAAAAAAH|      2000-03-03|          NULL|           71446|
|         8|AAAAAAAAAAAAAAAI|      2000-06-02|          NULL|        

In [13]:
# un magasin « fermé » : à quelle date ?
fermes = sto.filter("s_closed_date_sk IS NOT NULL").select("s_store_sk", "s_store_id", F.col("s_closed_date_sk").cast("int").alias("k"))
fermes.join(d, fermes.k == d.d_date_sk, "left").select("s_store_sk", "s_store_id", "k", "d_date").show()

+----------+----------------+-----+----------+
|s_store_sk|      s_store_id|    k|    d_date|
+----------+----------------+-----+----------+
|         0|AAAAAAAAAAAAAAAA|71238|2095-01-16|
|         1|AAAAAAAAAAAAAAAB|18994|1952-01-03|
|         2|AAAAAAAAAAAAAAAC| 8900|1924-05-15|
|         3|AAAAAAAAAAAAAAAD|69902|2091-05-21|
|         4|AAAAAAAAAAAAAAAE|79617|2117-12-26|
|         5|AAAAAAAAAAAAAAAF|88139|2141-04-26|
|         6|AAAAAAAAAAAAAAAG|15761|1943-02-26|
|         7|AAAAAAAAAAAAAAAH|71446|2095-08-12|
|         8|AAAAAAAAAAAAAAAI|50689|2038-10-13|
|         9|AAAAAAAAAAAAAAAJ|25131|1968-10-22|
|        10|AAAAAAAAAAAAAAAK| 6235|1917-01-27|
|        11|AAAAAAAAAAAAAAAL|43634|2019-06-20|
+----------+----------------+-----+----------+



## 7. Sauvegarde des résultats (preuve) et arrêt

In [14]:
resume = {"date": str(__import__("datetime").datetime.now()), "spark": spark.version, "source": C.RAW,
          "profils": profils, "controles_store_sales": controles_ss, "integrite": integrite,
          "doublons_cle_store_sales": {"groupes": nb_groupes, "lignes": nb_lignes_dup, "doublons_exacts": exacts},
          "tukey_net_paid": {"q1": q1, "q3": q3, "seuil": seuil, "lignes": atyp}}
print(C.save_json(resume, "03_exploration_resultats.json"))
spark.stop()

/mnt/c/Users/meddy/Desktop/Data Lake Warehouse Lakehouse/J3/J3_Retail_Intelligence/preuves/03_exploration_resultats.json
